# Fraud Shield AI

**Enterprise Fraud Detection on Highly Imbalanced Transactions**

Authoring goal: production-grade ML prototype suitable for architect-level review.

---

## Executive Summary
This notebook builds and evaluates an end-to-end fraud detection system on an extremely imbalanced credit-card dataset. We compare baseline, sampling, class-weight, and threshold-optimization strategies across Logistic Regression, Random Forest, and XGBoost.

The evaluation is intentionally **PR-AUC and F2 driven**, because in fraud operations missing a fraud event (false negative) is usually much more expensive than reviewing a non-fraud alert (false positive).

## 1. Problem Statement

Fraud detection in card transactions is a classic rare-event problem:
- Fraud class prevalence is tiny.
- Distribution shift is frequent due to changing attack patterns.
- Naive accuracy can be misleadingly high.

Business objective:
1. Maximize fraud capture (recall) with stable precision.
2. Optimize threshold based on operational tolerance.
3. Keep model and pipeline reproducible, explainable, and deployable.

## 2. Dataset Choice (Kaggle, as of May 2026)

Selected dataset: **Credit Card Fraud Detection** (`mlg-ulb/creditcardfraud`)

- Link: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
- Size: 284,807 rows
- Frauds: 492
- Fraud prevalence: 0.172%
- Imbalance ratio: ~577:1

### Why this dataset
- It remains the most established public benchmark on Kaggle for *extreme* imbalance in credit-card fraud detection.
- It includes realistic transaction behavior (anonymized PCA features + amount + time).
- The dataset card itself recommends precision-recall-centric evaluation due severe imbalance.

### Alternatives considered
- IEEE-CIS Fraud Detection: larger and richer, but broader ecommerce identity context and lower imbalance severity relative to this benchmark objective.
- Newer synthetic datasets: useful for augmentation, but less preferred as primary benchmark for this assignment.

In [ ]:
# Core imports and configuration
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from src.config import KAGGLE_DATASET_URL, KAGGLE_DATASET_SLUG
from src.eda import run_full_eda
from src.evaluation import (
    evaluate_threshold_grid,
    evaluate_train_valid_test,
    flatten_metrics_for_table,
    select_best_threshold,
)
from src.modeling import generate_experiment_specs, predict_probabilities, train_with_random_search
from src.preprocessing import (
    TARGET_COLUMN,
    assess_data_quality,
    drop_duplicates,
    load_dataset,
    stratified_train_valid_test_split,
    summarize_class_balance,
)
from src.utils import save_dataframe, set_global_seed
from src.visualization import (
    plot_confusion_matrix,
    plot_feature_importance,
    plot_model_comparison_dashboard,
    plot_precision_recall_curves,
    plot_roc_curves,
    plot_threshold_tradeoff,
    try_plot_shap_summary,
)

warnings.filterwarnings("ignore")
set_global_seed(42)

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data/raw/creditcard.csv"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
PLOTS_DIR = OUTPUT_ROOT / "plots"
METRICS_DIR = OUTPUT_ROOT / "metrics"
REPORTS_DIR = OUTPUT_ROOT / "reports"

for d in [PLOTS_DIR, METRICS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Kaggle dataset slug:", KAGGLE_DATASET_SLUG)
print("Kaggle dataset URL:", KAGGLE_DATASET_URL)
print("Expected local path:", DATA_PATH)

## 3. Data Loading and Quality Validation

Engineering principles applied:
- explicit schema validation
- duplicate detection and removal
- audit-friendly quality reporting
- fail-fast behavior if target labels are invalid

This prevents hidden data issues from silently invalidating model metrics.

In [ ]:
# Dataset source comment for traceability:
# https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

df = load_dataset(DATA_PATH, target_col=TARGET_COLUMN)
quality = assess_data_quality(df)

print("Initial shape:", df.shape)
print("Duplicates:", quality.duplicate_rows)
print("Missing values total:", quality.missing_values_total)

# Remove exact duplicate rows as a conservative data cleaning step
df, removed_dupes = drop_duplicates(df)
print("Rows removed due to duplication:", removed_dupes)
print("Shape after deduplication:", df.shape)

## 4. Class Imbalance Severity

Why imbalance is dangerous:
- A classifier can achieve excellent accuracy by almost never predicting fraud.
- Fraud operations care more about catching positives than maximizing raw hit-rate on negatives.

Why PR-AUC matters more than ROC-AUC here:
- ROC can remain high even with weak positive precision under heavy imbalance.
- PR-AUC directly reflects positive-class ranking quality under low prevalence.

In [ ]:
class_summary = summarize_class_balance(df[TARGET_COLUMN])
class_summary

In [ ]:
baseline_accuracy = (df[TARGET_COLUMN] == 0).mean()
print(f"Always-predict-non-fraud accuracy: {baseline_accuracy:.4%}")
print("This demonstrates why accuracy alone is misleading for fraud detection.")

## 5. Detailed EDA

The EDA module exports presentation-ready visuals:
- class distribution
- amount analysis
- time pattern analysis
- correlation heatmap
- feature distribution contrasts
- skewness profile

These plots are saved to `outputs/plots/` and reused in reports/slides.

In [ ]:
eda_stats = run_full_eda(df, plots_dir=PLOTS_DIR, reports_dir=REPORTS_DIR, target_col=TARGET_COLUMN)
eda_stats

In [ ]:
# Preview generated EDA files
from IPython.display import display, Image

eda_plot_files = [
    "eda_class_distribution.png",
    "eda_amount_distribution.png",
    "eda_time_analysis.png",
    "eda_correlation_heatmap.png",
    "eda_feature_distributions.png",
    "eda_skewness_analysis.png",
]

for file_name in eda_plot_files:
    path = PLOTS_DIR / file_name
    if path.exists():
        print(path)
        display(Image(filename=str(path), width=900))

## 6. Data Splitting and Leakage Prevention

Leakage controls:
- stratified train/validation/test split
- test set held untouched until final evaluation
- resampling happens inside imbalanced-learn pipelines during fitting only

This is critical to avoid optimistic fraud metrics.

In [ ]:
split = stratified_train_valid_test_split(
    df,
    target_col=TARGET_COLUMN,
    test_size=0.20,
    valid_size=0.20,
    random_state=42,
)

print("X_train:", split.X_train.shape, "Fraud rate:", split.y_train.mean())
print("X_valid:", split.X_valid.shape, "Fraud rate:", split.y_valid.mean())
print("X_test:", split.X_test.shape, "Fraud rate:", split.y_test.mean())

## 7. Modeling Design

### Imbalance techniques compared
1. Baseline
2. Random undersampling
3. Random oversampling
4. SMOTE
5. SMOTE + Tomek Links
6. Class weights
7. Validation threshold tuning

### Models compared
1. Logistic Regression
2. Random Forest
3. XGBoost

### Optimization strategy
- Hyperparameter tuning via `RandomizedSearchCV`
- CV scorer set includes PR-AUC, ROC-AUC, and F2
- Refit objective is PR-AUC

In [ ]:
experiment_specs = generate_experiment_specs()
print("Total experiments:", len(experiment_specs))
experiment_specs[:5]

## 8. Train and Evaluate All Experiments

This is the core enterprise comparison loop.

For each experiment:
- fit model with CV tuning
- score train/validation/test using probabilities
- tune threshold on validation for F2
- record full metric profile and business proxy cost
- save threshold tables and diagnostics plots

In [ ]:
# You can reduce compute for quick checks:
# experiment_specs = experiment_specs[:4]

comparison_rows = []
roc_pr_payloads = []
trained_models = {}

for i, spec in enumerate(experiment_specs, start=1):
    experiment_name = f"{spec.model_name}__{spec.sampling_strategy}__{'class_weighted' if spec.use_class_weight else 'no_class_weight'}"
    print(f"[{i}/{len(experiment_specs)}] {experiment_name}")

    trained = train_with_random_search(
        spec=spec,
        X_train=split.X_train,
        y_train=split.y_train,
        random_state=42,
        n_iter=12,
        cv_folds=4,
    )

    prob_train = predict_probabilities(trained.best_estimator, split.X_train)
    prob_valid = predict_probabilities(trained.best_estimator, split.X_valid)
    prob_test = predict_probabilities(trained.best_estimator, split.X_test)

    threshold_table = evaluate_threshold_grid(
        y_true=split.y_valid,
        y_prob=prob_valid,
        beta=2.0,
        cost_false_positive=1.0,
        cost_false_negative=25.0,
    )

    threshold_choice = select_best_threshold(
        threshold_table,
        objective="f2",
        min_precision=0.0,
    )

    split_metrics = evaluate_train_valid_test(
        y_train=split.y_train,
        prob_train=prob_train,
        y_valid=split.y_valid,
        prob_valid=prob_valid,
        y_test=split.y_test,
        prob_test=prob_test,
        tuned_threshold=threshold_choice.threshold,
        beta=2.0,
        cost_false_positive=1.0,
        cost_false_negative=25.0,
    )

    row = flatten_metrics_for_table(
        experiment_name=experiment_name,
        split_metrics=split_metrics,
        best_cv_pr_auc=trained.best_cv_pr_auc,
        best_cv_roc_auc=trained.cv_roc_auc,
        best_cv_f2=trained.cv_f2,
    )
    row["model_name"] = spec.model_name
    row["sampling_strategy"] = spec.sampling_strategy
    row["use_class_weight"] = spec.use_class_weight
    row["best_threshold"] = threshold_choice.threshold
    row["best_params"] = str(trained.best_params)

    comparison_rows.append(row)
    trained_models[experiment_name] = trained.best_estimator

    save_dataframe(threshold_table, METRICS_DIR / f"threshold_table_{experiment_name}.csv", index=False)

    y_pred_test = (prob_test >= threshold_choice.threshold).astype(int)
    plot_confusion_matrix(
        y_true=split.y_test.values,
        y_pred=y_pred_test,
        output_path=PLOTS_DIR / f"confusion_matrix_{experiment_name}.png",
        title=f"Confusion Matrix | {experiment_name}",
    )
    plot_threshold_tradeoff(
        threshold_table=threshold_table,
        output_path=PLOTS_DIR / f"threshold_tradeoff_{experiment_name}.png",
        title=f"Threshold Tradeoff | {experiment_name}",
    )

    roc_pr_payloads.append({
        "name": experiment_name,
        "y_true": split.y_test.values,
        "y_prob": prob_test,
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("test_pr_auc", ascending=False).reset_index(drop=True)
save_dataframe(comparison_df, METRICS_DIR / "model_comparison_table.csv", index=False)
comparison_df.head(10)

## 9. Results Dashboard

We focus on:
- PR-AUC ranking quality
- F2 capture orientation
- FP/FN cost proxy
- threshold-dependent precision-recall tradeoff

In [ ]:
plot_model_comparison_dashboard(comparison_df, PLOTS_DIR / "dashboard_top_experiments_test_pr_auc.png", metric="test_pr_auc")
plot_model_comparison_dashboard(comparison_df, PLOTS_DIR / "dashboard_top_experiments_test_f2.png", metric="test_f2")

from IPython.display import Image, display

display(Image(filename=str(PLOTS_DIR / "dashboard_top_experiments_test_pr_auc.png"), width=1000))
display(Image(filename=str(PLOTS_DIR / "dashboard_top_experiments_test_f2.png"), width=1000))

In [ ]:
top_experiments = set(comparison_df.head(8)["experiment"].tolist())
top_payloads = [p for p in roc_pr_payloads if p["name"] in top_experiments]

plot_roc_curves(top_payloads, PLOTS_DIR / "roc_curve_top_experiments.png")
plot_precision_recall_curves(top_payloads, PLOTS_DIR / "pr_curve_top_experiments.png")

display(Image(filename=str(PLOTS_DIR / "roc_curve_top_experiments.png"), width=900))
display(Image(filename=str(PLOTS_DIR / "pr_curve_top_experiments.png"), width=900))

## 10. Champion Model Interpretation

Select the best experiment by test PR-AUC, then inspect:
- feature importance
- confusion matrix behavior
- threshold profile
- optional SHAP explanation

In [ ]:
champion_row = comparison_df.iloc[0]
champion_name = champion_row["experiment"]
champion_model = trained_models[champion_name]

print("Champion:", champion_name)
print(champion_row[["test_pr_auc", "test_recall", "test_precision", "test_f2", "test_cost"]])

feature_importance_df = plot_feature_importance(
    champion_model,
    feature_names=split.X_train.columns.tolist(),
    output_path=PLOTS_DIR / f"feature_importance_{champion_name}.png",
    title=f"Feature Importance | {champion_name}",
    top_n=20,
)

if not feature_importance_df.empty:
    save_dataframe(feature_importance_df, METRICS_DIR / f"feature_importance_{champion_name}.csv", index=False)

shap_ok = try_plot_shap_summary(
    champion_model,
    X_sample=split.X_test,
    output_path=PLOTS_DIR / f"shap_summary_{champion_name}.png",
)
print("SHAP generated:", shap_ok)

In [ ]:
if (PLOTS_DIR / f"feature_importance_{champion_name}.png").exists():
    display(Image(filename=str(PLOTS_DIR / f"feature_importance_{champion_name}.png"), width=900))

if (PLOTS_DIR / f"shap_summary_{champion_name}.png").exists():
    display(Image(filename=str(PLOTS_DIR / f"shap_summary_{champion_name}.png"), width=900))

## 11. Architect-Level Analysis

### Why fraud detection is difficult
- Class imbalance dilutes positive signal.
- Adversaries adapt behavior quickly.
- Label delays can slow supervised retraining.
- Customer friction constraints limit aggressive policies.

### Precision vs Recall tradeoff
- High recall increases fraud capture but can increase false alerts.
- High precision reduces review load but may miss fraud.
- Threshold tuning operationalizes this tradeoff according to risk appetite.

### Why F2 in this project
F2 gives more weight to recall than precision, which aligns with environments where missing fraudulent activity has materially higher financial impact than incremental review cost.

### PR-AUC superiority here
PR-AUC is sensitive to positive-class performance under low prevalence; it is therefore more trustworthy than accuracy and often more decision-relevant than ROC-AUC in extreme-imbalance detection.

### Overfitting checks
The notebook and pipeline compare train/validation/test metrics for each experiment. Significant train-test gaps indicate potential overfitting and should trigger regularization or complexity controls.

### Latency vs accuracy tradeoff
- Logistic regression: fastest, most interpretable.
- Random forest: moderate latency and strong baseline.
- XGBoost: typically best rank performance with higher compute.

### Real-world deployment limitations
- Static offline splits do not fully represent streaming drift.
- Need temporal backtesting and online monitoring for production assurance.
- Fraud pattern evolution requires continuous adaptation and policy governance.

## 12. Business Recommendations

1. Use PR-AUC + F2 + cost proxy as core selection criteria.
2. Tune thresholds by analyst capacity and fraud-loss tolerance.
3. Monitor FN-driven loss and FP-driven customer friction jointly.
4. Establish drift alerts and scheduled retraining governance.
5. Pair model scoring with rule-engine and analyst feedback loop.

## 13. Final Conclusion

This implementation demonstrates an enterprise-ready fraud detection workflow with:
- modular architecture
- leakage-safe imbalance handling
- robust threshold optimization
- business-aligned metrics
- explainability artifacts

The most suitable production candidate is the model/strategy pair that achieves the best balance between PR-AUC, recall-weighted F2, and expected operational cost on held-out test data.

## 14. Future Improvements

- Add temporal walk-forward validation.
- Add probability calibration and reject option policy.
- Add online feature-store + streaming inference design.
- Add concept drift detectors (population + label-shift).
- Add MLOps controls (model registry, lineage, rollback playbooks).